# T34 - 债券新发行

## 第1章：项目概述与数据字典

---

### 项目概述

债券新发行系统是一个综合性的债券一级市场数据采集和分析工具，主要功能包括：

1. **债券到期数据采集**：从Wind API获取各种类型债券的发行和到期数据
2. **债券新发行分析**：从同花顺iFinD API获取新发债券的详细信息
3. **数据存储与管理**：将数据存储到MySQL数据库，支持动态列管理
4. **数据去重与清洗**：自动识别和删除重复数据

### 项目定位

**债券发行与到期数据采集系统**

| 模块 | 数据源 | 功能 | 输出表 |
|------|--------|------|--------|
| zqdq.py | Wind API | 债券到期数据采集 | 债券到期 |
| zqfx.py | 同花顺iFinD | 债券新发行数据采集 | 债券新发行1 |

### 数据流向图

```
zqdq.py (债券到期):
  Wind API -> 27次调用/天 -> 列名转换 -> 入库 -> 表: 债券到期

zqfx.py (债券新发行):
  同花顺API -> 单次调用 -> 列名转换 -> 入库 -> 去重 -> 表: 债券新发行1
```

### 执行流程对比

| 对比项 | zqdq.py (到期) | zqfx.py (新发) |
|-------|----------------|----------------|
| 数据源 | Wind | 同花顺 |
| 时间范围 | 未来7天 | 未来30天 |
| API函数 | w.wset() | THS_DR() |
| 债券类型 | 27类 | 通过zqlx参数控制 |
| 是否去重 | 否 | 是（债券名称去重） |
| 循环次数 | 按日期循环（最多7次） | 单次批量调用 |

### 数据字典

#### 1. 债券到期表 (yq.债券到期)

| 字段名 | 类型 | 说明 |
|--------|------|------|
| dt | DATE | 发行日期 |
| totalredemption | DECIMAL | 总兑付金额（亿元） |
| bondtype | VARCHAR(50) | 债券类型 |
| isurbaninvestmentbonds | VARCHAR(10) | 是否城投债 |

#### 2. 债券新发行表 (yq.债券新发行1)

| 字段名 | 类型 | 说明 |
|--------|------|------|
| trade_code | VARCHAR(20) | 债券代码 |
| sec_name | VARCHAR(100) | 债券名称 |
| dt | DATE | 发行日期 |
| planissueamount | DECIMAL | 计划发行规模（亿元） |
| bondterm | VARCHAR(50) | 债券期限 |
| bondtype | VARCHAR(50) | 债券类型 |
| isurbaninvestmentbonds | VARCHAR(10) | 是否城投债 |

### 债券类型分类

#### Wind API 债券类型（共27类）

| 序号 | Wind类型 | 中文名称 | 是否分城投 |
|------|---------|---------|----------|
| 1 | governmentbonds | 国债 | 否 |
| 2 | centralbankbills | 央行票据 | 否 |
| 3 | cds | 存单 | 否 |
| 4 | commercialbankbonds | 普通金融债 | 否 |
| 5 | policybankbonds | 政策银行债 | 否 |
| 6 | commercialbanksubordinatedbonds | 商业银行次级债券 | 否 |
| 7 | insurancecompanybonds | 保险债 | 否 |
| 8 | corporatebondsofsecuritiescompany | 证券公司债 | 否 |
| 9 | securitiescompanycps | 证券公司短融债 | 否 |
| 10 | otherfinancialagencybonds | 其他金融机构债 | 否 |
| 11 | enterprisebonds | 企业债 | 是/否 |
| 12 | corporatebonds | 公司债 | 是/否 |
| 13 | medium-termnotes | 中期票据 | 是/否 |
| 14 | cps | 短期融资券 | 是/否 |
| 15 | projectrevenuenotes | 项目收益票据 | 否 |
| 16 | ppn | PPN | 是/否 |
| 17 | supranationalbonds | 国际机构债 | 否 |
| 18 | agencybonds | 政府支持机构债 | 否 |
| 19 | standardizednotes | 标准化票据 | 否 |
| 20 | abs | ABS | 否 |
| 21 | convertiblebonds | 可转债 | 否 |
| 22 | exchangeablebonds | 可交换债 | 否 |
| 23 | detachableconvertiblebonds | 可分离转债 | 否 |

### API接口说明

#### Wind API (w.wset)

**接口名称**: bondissuanceandmaturity

**主要参数**:
- `startdate`/`enddate`: 查询日期范围
- `datetype=startdate`: 按发行日查询
- `duedatetype=byactualpaymentdate`: 按实际到期日
- `bondtype`: 债券类型
- `conceptbond`: 是否为城投债
- `field=startdate,enddate,totalredemption`: 返回字段

#### 同花顺iFinD API (THS_DR)

**接口代码**: p04524

**主要参数**:
- `sdate`/`edate`: 日期范围
- `zqlx`: 债券类型代码列表
- `datetype=5`: 按发行日

**返回字段**:
- `jydm`: 债券代码
- `jydm_mc`: 债券名称
- `p04524_f005`: 发行日期
- `p04524_f006`: 计划发行量
- `p04524_f009`: 期限
- `p04524_f029`: 类型
- `p04524_f063`: 是否城投

In [ ]:
# 环境验证
import sys
print(f"Python版本: {sys.version}")
print(f"Python路径: {sys.executable}")

In [ ]:
# 验证必要的库
import pandas as pd
import sqlalchemy
from datetime import datetime
print("核心库导入成功")
print(f"Pandas版本: {pd.__version__}")
print(f"SQLAlchemy版本: {sqlalchemy.__version__}")

### 项目执行流程

```
zqdq.py (债券到期):
  1. 初始化数据库连接
  2. 启动Wind API
  3. 确定日期范围（未来7天）
  4. 循环处理每个日期:
     - 查询数据库已有日期
     - 对每种债券类型调用Wind API
     - 标记债券类型和城投标识
     - 列名转换（中→英）
     - 插入数据库
     - 列名恢复（英→中）

zqfx.py (债券新发行):
  1. 初始化数据库连接
  2. 登录同花顺iFinD
  3. 确定日期范围（未来1-30天）
  4. 调用iFinD API获取数据
  5. 列名转换和数据入库
  6. 执行去重SQL
```

---

**下一章**: [2_环境配置与初始化](2_环境配置与初始化.ipynb)